# 35. Native Multi-Token Prediction｜原生多 Token 预测

**难度：** Hard | **环境：** CPU + GPU | **标签：** `推理优化`, `投机解码`, `MTP`

## 本节导读

原生多 Token 预测（MTP）把未来位置的预测模块放在目标模型内部，用共享状态形成多个候选，不再依赖独立草稿模型。本节关注内置候选如何产生、怎样与目标模型结果逐位置验证，以及连续接受长度如何决定单轮有效推进量。


## 前置阅读

- [Part 02 · 21 解码策略](./21_Decoding_Strategies.ipynb)
- [Part 02 · 23 投机解码](./23_Speculative_Decoding.ipynb)


### Step 1：内置预测模块如何形成候选

原生 MTP 的候选来自目标模型检查点内部。多个预测深度复用主模型状态，分别面向更远的未来位置给出词表分数；每个深度选择一个候选后，按位置组成有序候选块。模型检查点没有相应模块时，运行参数不能凭空启用这条路径。

| 对象 | 作用 | 必须成立的条件 |
| --- | --- | --- |
| 主模型状态 | 提供当前上下文表示 | 与本轮目标验证共享同一模型语义 |
| MTP 预测深度 | 面向不同未来位置给出词表分数 | 检查点包含对应数量的内置模块 |
| 有序候选块 | 按预测深度排列候选 Token | 顺序与待验证位置一一对应 |

![原生 MTP 与投机解码机制链](../docs/public/02_PyTorch_Algorithms/35_multi_token_overview.svg)


### Step 2：候选怎样与目标结果逐位置验证

目标模型为候选位置给出验证结果。贪心路径下，第 $i$ 个 MTP 候选与目标模型在同一位置的首选 Token 比较：相同则进入连续接受前缀；首次不同则关闭后缀，并使用目标模型在该位置的结果修正本轮输出。

| 验证状态 | 条件 | 状态变化 |
| --- | --- | --- |
| 当前位置匹配 | MTP 候选等于目标首选 Token | 接受长度加一，继续检查 |
| 首次不匹配 | 两者在当前位置不同 | 提交已接受前缀和目标修正 Token |
| 全部匹配 | 所有候选均通过 | 提交候选块和额外目标 Token |


### Step 3：连续接受长度如何转化为有效推进

候选数量只表示最多尝试多少位置，连续接受长度才决定本轮能复用多少候选。更长的候选块扩大潜在推进量，也会增加候选计算和验证成本；首次不匹配越早，后部候选越难转化为实际收益。

| 观察量 | 表示的机制 | 不能单独推出的结论 |
| --- | --- | --- |
| 候选长度 | 本轮最大尝试范围 | 不等于实际推进量 |
| 连续接受长度 | 可直接提交的候选数量 | 不等于端到端加速比 |
| 每轮输出长度 | 接受前缀加修正或追加 Token | 仍需结合候选和验证成本 |

![多 Token 解码的接受长度与收益](../docs/public/02_PyTorch_Algorithms/35_acceptance_budget.svg)


### Step 4：实现原生 MTP 候选与贪心验证

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1 · `select_mtp_candidates` | 从各预测深度形成有序候选块 | 每个深度独立选择；候选数不超过预算 | 正常候选、预算截断、空输入与维度错误 |
| TODO 2 · `continuous_match_length` | 计算候选与目标结果的连续匹配长度 | 首次不匹配立即停止 | 首位置失败、中途失败与全部匹配 |
| TODO 3 · `build_mtp_round` | 根据匹配边界提交修正或追加 Token | 后缀不得越过首次不匹配位置 | 修正路径、全部匹配路径与统一状态记录 |


In [ ]:
from typing import List, Sequence
import torch


In [ ]:
def _validate_depth_logits(depth_logits: torch.Tensor) -> torch.Tensor:
    """校验 MTP 预测深度与词表维度。"""
    scores = torch.as_tensor(depth_logits)
    if scores.ndim != 2 or scores.shape[0] == 0 or scores.shape[1] == 0:
        raise ValueError("depth_logits 必须是非空二维张量 [depth, vocab]")
    if not torch.isfinite(scores).all():
        raise ValueError("词表分数不能包含 NaN 或 Inf")
    return scores


def select_mtp_candidates(depth_logits: torch.Tensor, max_candidates: int) -> List[int]:
    """按预测深度形成受预算约束的有序候选块。"""
    scores = _validate_depth_logits(depth_logits)
    if not isinstance(max_candidates, int) or isinstance(max_candidates, bool) or max_candidates < 1:
        raise ValueError("max_candidates 必须是正整数")
    # ==========================================
    # TODO 1：从各预测深度形成有序候选块。
    # ==========================================
    # candidates = ???  # 各深度最高分 Token，长度不超过 max_candidates。
    raise NotImplementedError("TODO 1：请形成 MTP 候选块")
    return candidates


def continuous_match_length(
    candidates: Sequence[int],
    target_tokens: Sequence[int],
) -> int:
    """返回候选与目标结果从首位置开始连续匹配的长度。"""
    if len(target_tokens) < len(candidates):
        raise ValueError("目标结果必须覆盖所有候选位置")
    accepted_length = 0
    for candidate, target in zip(candidates, target_tokens):
        # ==========================================
        # TODO 2：在首次不匹配位置停止连续接受。
        # ==========================================
        # matches = ???  # 当前 MTP 候选与目标首选 Token 相同时为 True。
        raise NotImplementedError("TODO 2：请比较当前位置")
        if not matches:
            break
        accepted_length += 1
    return accepted_length


def build_mtp_round(candidates: Sequence[int], target_tokens: Sequence[int]) -> dict:
    """根据连续匹配边界形成本轮输出和下一轮状态。"""
    if len(target_tokens) != len(candidates) + 1:
        raise ValueError("目标结果必须包含候选位置和一个额外位置")
    accepted_length = continuous_match_length(candidates, target_tokens)
    # ==========================================
    # TODO 3：提交接受前缀，并追加修正或额外目标 Token。
    # ==========================================
    # emitted_tokens = ???  # 接受前缀后追加 target_tokens[accepted_length]。
    # all_candidates_accepted = ???  # 接受长度等于候选长度时为 True。
    raise NotImplementedError("TODO 3：请形成本轮输出")
    return {
        "candidate_count": len(candidates),
        "accepted_length": accepted_length,
        "emitted_tokens": emitted_tokens,
        "all_candidates_accepted": all_candidates_accepted,
        "first_mismatch": None if all_candidates_accepted else accepted_length,
    }


### 测试

测试分别检查内置候选形成、连续匹配边界以及修正和追加两条状态路径。


In [ ]:
def test_select_mtp_candidates():
    """候选应按预测深度排列，并受最大候选数限制。"""
    logits = torch.tensor([[0.1, 0.9, 0.0], [0.8, 0.1, 0.1], [0.2, 0.3, 0.5]])
    assert select_mtp_candidates(logits, 2) == [1, 0]
    assert select_mtp_candidates(logits, 8) == [1, 0, 2]
    for invalid in (torch.empty(0, 3), torch.tensor([0.1, 0.9])):
        try:
            select_mtp_candidates(invalid, 2)
        except ValueError:
            pass
        else:
            raise AssertionError("空输入或非二维输入应失败")


def test_continuous_match_length():
    """连续匹配必须在首个不同位置停止。"""
    assert continuous_match_length([4, 5, 6], [4, 5, 9, 8]) == 2
    assert continuous_match_length([4, 5], [9, 5, 8]) == 0
    assert continuous_match_length([4, 5], [4, 5, 8]) == 2


def test_build_mtp_round():
    """中途不匹配使用目标修正，全部匹配使用额外目标 Token。"""
    rejected = build_mtp_round([4, 5, 6], [4, 5, 9, 8])
    assert rejected == {
        "candidate_count": 3,
        "accepted_length": 2,
        "emitted_tokens": [4, 5, 9],
        "all_candidates_accepted": False,
        "first_mismatch": 2,
    }
    accepted = build_mtp_round([4, 5], [4, 5, 8])
    assert accepted["emitted_tokens"] == [4, 5, 8]
    assert accepted["all_candidates_accepted"] is True
    assert accepted["first_mismatch"] is None


test_select_mtp_candidates()
test_continuous_match_length()
test_build_mtp_round()
print("✅ 原生 MTP 候选与贪心验证测试通过")


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难，再向下查看参考代码与解析。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def _validate_depth_logits(depth_logits: torch.Tensor) -> torch.Tensor:
    """校验 MTP 预测深度与词表维度。"""
    scores = torch.as_tensor(depth_logits)
    if scores.ndim != 2 or scores.shape[0] == 0 or scores.shape[1] == 0:
        raise ValueError("depth_logits 必须是非空二维张量 [depth, vocab]")
    if not torch.isfinite(scores).all():
        raise ValueError("词表分数不能包含 NaN 或 Inf")
    return scores


# ==========================================
# TODO 1：从各预测深度形成有序候选块。
# ==========================================
def select_mtp_candidates(depth_logits: torch.Tensor, max_candidates: int) -> List[int]:
    """按预测深度形成受预算约束的有序候选块。"""
    scores = _validate_depth_logits(depth_logits)
    if not isinstance(max_candidates, int) or isinstance(max_candidates, bool) or max_candidates < 1:
        raise ValueError("max_candidates 必须是正整数")
    candidates = scores.argmax(dim=-1)[:max_candidates].tolist()
    return candidates


# ==========================================
# TODO 2：在首次不匹配位置停止连续接受。
# ==========================================
def continuous_match_length(
    candidates: Sequence[int],
    target_tokens: Sequence[int],
) -> int:
    """返回候选与目标结果从首位置开始连续匹配的长度。"""
    if len(target_tokens) < len(candidates):
        raise ValueError("目标结果必须覆盖所有候选位置")
    accepted_length = 0
    for candidate, target in zip(candidates, target_tokens):
        matches = int(candidate) == int(target)
        if not matches:
            break
        accepted_length += 1
    return accepted_length


# ==========================================
# TODO 3：提交接受前缀，并追加修正或额外目标 Token。
# ==========================================
def build_mtp_round(candidates: Sequence[int], target_tokens: Sequence[int]) -> dict:
    """根据连续匹配边界形成本轮输出和下一轮状态。"""
    if len(target_tokens) != len(candidates) + 1:
        raise ValueError("目标结果必须包含候选位置和一个额外位置")
    accepted_length = continuous_match_length(candidates, target_tokens)
    emitted_tokens = list(candidates[:accepted_length]) + [int(target_tokens[accepted_length])]
    all_candidates_accepted = accepted_length == len(candidates)
    return {
        "candidate_count": len(candidates),
        "accepted_length": accepted_length,
        "emitted_tokens": emitted_tokens,
        "all_candidates_accepted": all_candidates_accepted,
        "first_mismatch": None if all_candidates_accepted else accepted_length,
    }


### 解析

#### TODO 1：形成 MTP 候选块

- **机制：** 每个内置预测深度为一个未来位置提供词表分数，按深度顺序选择候选后形成候选块。
- **不变量：** 候选顺序不能重排，数量不得超过预测深度和本轮预算。
- **测试证据：** 测试覆盖正常候选、预算截断、预算大于深度数、空输入和维度错误。

#### TODO 2：计算连续匹配长度

- **机制：** 从首位置逐个比较 MTP 候选与目标首选 Token，首次不匹配即结束接受前缀。
- **不变量：** 接受结果始终是候选块的连续前缀，后部匹配不能越过前面的不匹配位置。
- **测试证据：** 测试覆盖首位置失败、中途失败和全部匹配。

#### TODO 3：形成本轮输出

- **机制：** 提交连续接受前缀，再加入目标模型在首个不匹配位置的修正 Token；全部匹配时加入额外目标 Token。
- **不变量：** 每轮只在接受前缀之后加入一个目标 Token，未验证的候选后缀不得进入输出。
- **测试证据：** 测试核对修正路径、全部匹配路径、首个不匹配位置和统一状态字段。


### Step 5：GPU 机制实验——原生 MTP 路径

CPU 题目确认内置候选、连续匹配和修正/追加状态；GPU 实验进一步检查模型检查点是否包含原生 MTP 层、后端是否命中对应路径，以及真实模型上的正确性、有效推进、延迟和显存。

| 学习问题 | CPU 已确认 | GPU 要确认 |
| --- | --- | --- |
| 原生 MTP 能否增加每次目标模型调用的有效推进量 | 候选形成、连续匹配和首个不匹配处理 | 同一模型检查点的仅目标模型/MTP-1 是否成功配对，以及输出、接受长度、Median/P95、吞吐和峰值显存 |

#### 5.1 环境与固定工作负载

本实验使用隔离的 `INF-VLLM-MTP` 环境。首次运行先完成[推理优化实验 Quickstart](../topic_discussion/inference_optimization/quickstart.md#inf-vllm-mtp-quickstart)；已有环境先通过同名 Profile 预检。只有仅目标模型路径完成模型加载、内核初始化、预热和固定输入生成后，才启用 MTP-1。

| 配置层 | 本节固定口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | `INF-VLLM-MTP`；隔离的 vLLM、PyTorch、CUDA wheel 与 Triton 组合 | 更换驱动、wheel 或 Triton 后使用新运行标签，并先重跑仅目标模型路径 |
| 12 GB 配置 | `Qwen/Qwen3.5-0.8B`；MTP-1；BF16；批量 1；`max_model_len=2048`；显存利用率 0.75 | 模型检查点必须包含原生 MTP 层；候选深度不得超过可用层数 |
| 24 GB 配置 | 先复用 0.8B 配对，再扩大上下文、并发或重复次数 | 9B BF16 不作为默认配置；更换模型后单独登记工作负载版本 |
| 配对变量 | 同一模型检查点的仅目标模型 → MTP-1 | 两条路径共用输入、数据类型、生成长度、随机种子和执行模式 |
| 测量方法 | 输出一致性、接受长度、Median/P95、吞吐、峰值显存和失败阶段 | 初始化和下载不计入正式延迟；两条路径分别预热 |
| 结果资产 | `benchmarks/results/inf_mtp_<run_label>.json` | 正式复测使用唯一标签；失败结果不得覆盖 |


In [1]:
RUN_MODE = 'real_backend'  # 默认 real_backend；只做配置检查时改为 dry_run。
RUN_LABEL = 'rtx5070ti_qwen35_08b_mtp1_v4'  # 默认不覆盖实测结果；真实复测改为唯一标签。
PROFILE_ID = 'INF-VLLM-MTP'  # 原生 MTP 使用独立 vLLM 环境。
SEMANTIC_ID = 'INF-MTP'  # 稳定机制身份。
WORKLOAD_ID = 'inf-native-mtp-qwen35-08b'  # 稳定工作负载身份。
WORKLOAD_VERSION = 'v2'  # v2 固定原生 MTP-1 与隔离 worker 配对协议。
MODEL_ID = 'Qwen/Qwen3.5-0.8B'  # 默认原生 MTP 小模型。
MODEL_REVISION = 'main'  # 运行时自动解析并记录不可变 commit。
DTYPE = 'bfloat16'  # baseline 与 candidate 共用 BF16。
MAX_NEW_TOKENS = 64  # 每条提示固定生成 64 Token。
MAX_MODEL_LEN = 2048  # 单卡机制实验只保留短上下文容量。
GPU_MEMORY_UTILIZATION = 0.75  # 为 Notebook 与 backend 保留显存余量。
ENFORCE_EAGER = False  # 两条路径共用 vLLM 默认编译/CUDA Graph 路径。
WARMUP_STEPS = 2  # 每条路径独立预热 2 次。
REPEAT_STEPS = 5  # 每条路径正式重复 5 次。
SPECULATIVE_TOKENS = 1  # checkpoint 只有一个原生 MTP 层。
SEED = 42  # 固定贪心生成与 backend 随机状态。
PROMPTS = [
    'Explain why speculative decoding can reduce latency in two concise paragraphs.',
    'Write a Python function that checks whether a string is a palindrome, then explain its complexity.',
    'A shop applies a 20 percent discount and then a 10 percent tax. Explain the final price for an item costing 100 dollars.',
]
OUTPUT_RELATIVE_PATH = f'02_PyTorch_Algorithms/benchmarks/results/inf_mtp_{RUN_LABEL}.json'


#### 5.2 执行真实后端配对并保存 JSON

本单元先检查模型配置中的原生 MTP 层，再用两个独立子进程顺序执行 target-only 与 MTP-1。初始化和模型下载不计入正式延迟；两条路径分别预热后才采集结果。


In [2]:
import importlib
from tools.project_runtime import resolve_project_root
import tools.mtp_mechanism_runtime as mtp_runtime

mtp_runtime = importlib.reload(mtp_runtime)
RESULT_PATH = resolve_project_root() / OUTPUT_RELATIVE_PATH
GPU_REPORT = mtp_runtime.run_native_mtp_benchmark(
    semantic_id=SEMANTIC_ID, profile_id=PROFILE_ID, run_mode=RUN_MODE, run_label=RUN_LABEL,
    workload_id=WORKLOAD_ID, workload_version=WORKLOAD_VERSION,
    model_id=MODEL_ID, model_revision=MODEL_REVISION, dtype=DTYPE, prompts=PROMPTS,
    max_new_tokens=MAX_NEW_TOKENS, max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION, enforce_eager=ENFORCE_EAGER,
    warmup_steps=WARMUP_STEPS, repeat_steps=REPEAT_STEPS,
    speculative_tokens=SPECULATIVE_TOKENS, seed=SEED, output_path=RESULT_PATH,
)
GPU_REPORT


{'schema_version': 'inference-mtp-mechanism/v1',
 'semantic_id': 'INF-MTP',
 'environment_profile': 'INF-VLLM-MTP',
 'run_mode': 'real_backend',
 'run_label': 'rtx5070ti_qwen35_08b_mtp1_v4',
 'workload': {'id': 'inf-native-mtp-qwen35-08b',
  'version': 'v2',
  'model_id': 'Qwen/Qwen3.5-0.8B',
  'requested_revision': 'main',
  'resolved_commit': '2fc06364715b967f1860aea9cf38778875588b17',
  'dtype': 'bfloat16',
  'prompt_count': 3,
  'max_new_tokens': 64,
  'max_model_len': 2048,
  'gpu_memory_utilization': 0.75,
  'enforce_eager': False,
  'warmup_steps': 2,
  'repeat_steps': 5,
  'speculative_tokens': 1,
  'seed': 42},
 'model_contract': {'model_type': 'qwen3_5',
  'text_model_type': 'qwen3_5_text',
  'mtp_num_hidden_layers': 1,
  'resolution_source': 'huggingface_hub_api'},
 'baseline': None,
 'candidate': None,
 'comparison': None,
 'failure': {'type': 'RuntimeError',
  'message': 'baseline worker failed (exit=1): serial_utils.py", line 508, in run_method\n    return func(*args, **k

#### 5.3 读取结果

读取固定身份、模型契约、两条路径和配对比较。`failure` 非空时先处理环境、模型或执行后端，不解释性能字段。


In [3]:
import json
from pathlib import Path

report = json.loads(Path(RESULT_PATH).read_text(encoding='utf-8'))
{
    'semantic_id': report.get('semantic_id'),
    'workload': report.get('workload'),
    'model_contract': report.get('model_contract'),
    'comparison': report.get('comparison'),
    'failure': report.get('failure'),
    'evidence_level': report.get('evidence_level'),
    'decision': report.get('decision'),
}


{'semantic_id': 'INF-MTP',
 'workload': {'id': 'inf-native-mtp-qwen35-08b',
  'version': 'v2',
  'model_id': 'Qwen/Qwen3.5-0.8B',
  'requested_revision': 'main',
  'resolved_commit': '2fc06364715b967f1860aea9cf38778875588b17',
  'dtype': 'bfloat16',
  'prompt_count': 3,
  'max_new_tokens': 64,
  'max_model_len': 2048,
  'gpu_memory_utilization': 0.75,
  'enforce_eager': False,
  'warmup_steps': 2,
  'repeat_steps': 5,
  'speculative_tokens': 1,
  'seed': 42},
 'model_contract': {'model_type': 'qwen3_5',
  'text_model_type': 'qwen3_5_text',
  'mtp_num_hidden_layers': 1,
  'resolution_source': 'huggingface_hub_api'},
 'comparison': None,
 'failure': {'type': 'RuntimeError',
  'message': 'baseline worker failed (exit=1): serial_utils.py", line 508, in run_method\n    return func(*args, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File "/home/yulinlin/anaconda3/envs/llm_vllm_mtp_cu129/lib/python3.12/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context\n    return func

#### 5.4 解释结果并作出决策

RTX 5070 Ti 已完成一次真实后端尝试。Qwen3.5-0.8B 的模型配置和一个原生 MTP 层均已识别，但 target-only 在 vLLM 初始化 Triton RoPE 内核时失败，因此本次运行没有产生 baseline/candidate 配对结果。

失败发生在启用 MTP 之前，说明当前驱动、CUDA 12.9 wheel、vLLM 与 Triton 组合未通过执行准入；它既不是 MTP 性能结论，也不是显存不足结论。下一次复测应先让 target-only 完整生成固定输出，再启用 MTP-1。

| 观察项 | 读取字段 | 当前结果 | 解释与动作 |
| --- | --- | --- | --- |
| 模型契约 | `model_contract`、resolved commit | Qwen3.5-0.8B commit `2fc0636`；识别 1 个 MTP 层 | checkpoint 支持 MTP-1 |
| 后端启动 | `failure`、`evidence_level` | Triton RoPE 内核报 `device kernel image is invalid` | target-only 未通过执行准入，保留失败环境快照 |
| 正确性与推进 | `comparison`、接受统计 | 未产生 | 修复执行栈后重新生成配对证据 |
| 性能与资源 | Median/P95、吞吐、峰值显存 | 未产生 | 不使用初始化失败耗时推断性能 |
| 证据与决策 | `evidence_level`、`decision` | `failed_real_backend_attempt`；`reject_until_fixed` | 在兼容环境从 target-only 开始复测 |


## 相关阅读

- [DeepSeek-V3：Multi-Token Prediction](https://arxiv.org/abs/2412.19437)
- [vLLM Speculators：MTP](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/algorithms/decision_guide/)
- [Part 02 · INF-EAGLE 特征草稿器与候选树](./INF-EAGLE_Feature_Drafter_and_Candidate_Tree.ipynb)
- [Part 02 · INF-PARALLEL-DRAFTING 并行草稿与置信度调度](./INF-PARALLEL-DRAFTING_DFlash_and_DSpark.ipynb)
